In [33]:
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)

In [2]:
##variables 
url = "https://gbfs.mex.lyftbikes.com/gbfs/es/station_status.json"

In [43]:
try:
    raw = pd.read_json(url)
    last_updated = raw["last_updated"].iloc[0]
    stations = pd.json_normalize(raw["data"]["stations"])
    stations['last_updated']= last_updated

    bool_cols = ["is_installed", "is_renting", "is_returning","is_charging",'eightd_has_available_keys']

    for col in bool_cols:
        stations[col] = stations[col].astype("boolean")

    stations["last_updated_ts"] = pd.to_datetime(stations["last_updated"], unit="s", utc=True)
    stations["last_reported_ts"] = pd.to_datetime(stations["last_reported"], unit="s", utc=True)
    run_id = stations["last_updated_ts"].iloc[0].strftime("%Y%m%d%H%M")
    stations["run_id"] = run_id

    stations = stations.drop(columns=["last_updated", "last_reported"])

except Exception as e:
    logging.error(f"Error en la extraccion de datos desde la fuente: {e}")

stations.head()

,station_id,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,is_installed,is_renting,is_returning,eightd_has_available_keys,is_charging,last_updated_ts,last_reported_ts,run_id
0,1,11,1,27,0,True,True,True,False,False,2026-05-29 22:36:34+00:00,2026-05-29 22:31:25+00:00,202605292236
1,5,4,0,15,0,True,True,True,False,False,2026-05-29 22:36:34+00:00,2026-05-29 22:22:31+00:00,202605292236
2,6,5,3,19,0,True,True,True,False,False,2026-05-29 22:36:34+00:00,2026-05-29 22:17:30+00:00,202605292236
3,7,0,0,0,0,True,False,False,False,False,2026-05-29 22:36:34+00:00,1970-01-02 00:00:00+00:00,202605292236
4,8,2,0,29,0,True,True,True,False,False,2026-05-29 22:36:34+00:00,2026-05-29 22:29:02+00:00,202605292236


In [46]:
import tempfile
from pathlib import Path
import boto3
import os

endpoint_url = os.environ["MINIO_ENDPOINT"] = "http://localhost:9000"
access_key = os.environ["MINIO_ROOT_USER"] = "minio"
secret_key = os.environ["MINIO_ROOT_PASSWORD"] = "minio1234"
trino_host = os.environ["TRINO_HOST"] = "localhost"
trino_port = os.environ["TRINO_PORT"] = "8090"
trino_user = os.environ["TRINO_USER"] = "root"

s3 = boto3.client(
        "s3",
        endpoint_url=endpoint_url,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name=os.getenv("AWS_REGION", "us-east-1"),
    )

tgt_bucket = "bck-bronze"
tgt_key = f"station_status/data/run_id={run_id}/station_status.parquet"

with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "station_status.parquet"
    stations.to_parquet(parquet_path, index=False, engine="pyarrow", compression="snappy")
    s3.upload_file(str(parquet_path), tgt_bucket, tgt_key)

In [36]:
raw.head()

,last_updated,ttl,data
stations,1780093783,10,"[{'station_id': '1', 'num_bikes_available': 10..."


In [37]:
print(stations.shape)
stations.head()

(677, 14)


,station_id,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,is_installed,is_renting,is_returning,eightd_has_available_keys,is_charging,last_updated_ts,last_reported_ts,last_updated_dt,run_id
0,1,10,1,28,0,True,True,True,False,False,2026-05-29 22:29:43+00:00,2026-05-29 22:27:56+00:00,2026-05-29,202605292229
1,5,4,0,15,0,True,True,True,False,False,2026-05-29 22:29:43+00:00,2026-05-29 22:22:31+00:00,2026-05-29,202605292229
2,6,5,3,19,0,True,True,True,False,False,2026-05-29 22:29:43+00:00,2026-05-29 22:17:30+00:00,2026-05-29,202605292229
3,7,0,0,0,0,True,False,False,False,False,2026-05-29 22:29:43+00:00,1970-01-02 00:00:00+00:00,2026-05-29,202605292229
4,8,2,0,29,0,True,True,True,False,False,2026-05-29 22:29:43+00:00,2026-05-29 22:29:02+00:00,2026-05-29,202605292229


In [38]:
stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 677 entries, 0 to 676
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   station_id                 677 non-null    object             
 1   num_bikes_available        677 non-null    int64              
 2   num_bikes_disabled         677 non-null    int64              
 3   num_docks_available        677 non-null    int64              
 4   num_docks_disabled         677 non-null    int64              
 5   is_installed               677 non-null    boolean            
 6   is_renting                 677 non-null    boolean            
 7   is_returning               677 non-null    boolean            
 8   eightd_has_available_keys  677 non-null    bool               
 9   is_charging                677 non-null    boolean            
 10  last_updated_ts            677 non-null    datetime64[ns, UTC]
 11  last_r

In [11]:
stations.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
station_id,677,677,702,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_bikes_available,677.0,NaN,NaN,NaN,5.74003,6.841425,0.0,1.0,4.0,8.0,63.0
num_bikes_disabled,677.0,NaN,NaN,NaN,2.152142,2.187691,0.0,1.0,2.0,3.0,14.0
num_docks_available,677.0,NaN,NaN,NaN,18.911374,9.369367,0.0,13.0,19.0,24.0,69.0
num_docks_disabled,677.0,NaN,NaN,NaN,0.017725,0.132049,0.0,0.0,0.0,0.0,1.0
is_installed,677.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
is_renting,677.0,NaN,NaN,NaN,0.995569,0.06647,0.0,1.0,1.0,1.0,1.0
is_returning,677.0,NaN,NaN,NaN,0.995569,0.06647,0.0,1.0,1.0,1.0,1.0
last_reported,677.0,NaN,NaN,NaN,1774833105.070901,96676401.831244,86400.0,1780091879.0,1780092247.0,1780092379.0,1780092451.0
eightd_has_available_keys,677,1,False,677,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
import os
from trino.dbapi import connect

conn = connect(
    host=os.getenv("TRINO_HOST", "localhost"),
    port=int(os.getenv("TRINO_PORT", "8090")),
    user=os.getenv("TRINO_USER", "root"),
    catalog="bronze",
    schema="ecobici",
    http_scheme="http",
)

cursor = conn.cursor()

schema_location = "s3a://bck-bronze/station_status/schema"
table_location = "s3a://bck-bronze/station_status/data"


In [19]:
create_schema_query = f"""
CREATE SCHEMA IF NOT EXISTS bronze.ecobici
WITH (LOCATION = '{schema_location}')
"""

create_table_query = f"""
CREATE TABLE IF NOT EXISTS bronze.ecobici.station_status (
    station_id varchar,
    num_bikes_available bigint,
    num_docks_available bigint,
    is_installed boolean,
    is_renting boolean,
    is_returning boolean,
    is_charging_station boolean,
    eightd_has_available_keys boolean,
    last_updated_ts timestamp(3),
    last_reported_ts timestamp(3),
    run_id varchar
)
WITH (
    external_location = '{table_location}',
    format = 'PARQUET',
    partitioned_by = ARRAY['run_id']
)
"""

In [21]:
create_table_query

"\nCREATE TABLE IF NOT EXISTS bronze.ecobici.station_status (\n    station_id varchar,\n    num_bikes_available bigint,\n    num_docks_available bigint,\n    is_installed boolean,\n    is_renting boolean,\n    is_returning boolean,\n    is_charging_station boolean,\n    eightd_has_available_keys boolean,\n    last_updated_ts timestamp(3),\n    last_reported_ts timestamp(3),\n    run_id varchar\n)\nWITH (\n    external_location = 's3a://bck-bronze/station_status/data',\n    format = 'PARQUET',\n    partitioned_by = ARRAY['run_id']\n)\n"

In [15]:
cursor.execute(create_schema_query)

In [20]:
cursor.execute(create_table_query)

In [22]:

# Hace visible en Trino la nueva partición escrita por Airflow
cursor.execute("CALL bronze.system.sync_partition_metadata('ecobici', 'station_status', 'ADD', true)")